In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import StringType, MapType

In [0]:
@dp.temporary_view(
    name="customers_bronze_view",
    comment="Standardized view of bronze customers with unified column names"
)
def customers_bronze_view():
    """
    Temporary view that standardizes customer data from bronze layer.
    Handles multiple column name variants using COALESCE to merge into standard schema.
    """
    df =  (
        spark.readStream.table("customers")
        .select(
            # Standardize customer_id from variants: customer_id, CustomerID, cust_id, customer_identifier
            coalesce(
                col("customer_id"),
                col("CustomerID"),
                col("cust_id"),
                col("customer_identifier")
            ).alias("customer_id"),
            
            # Standardize customer_email from variants: customer_email, email_address
            coalesce(
                col("customer_email"),
                col("email_address")
            ).alias("customer_email"),
            
            # Standardize customer_name from variants: customer_name, full_name
            coalesce(
                col("customer_name"),
                col("full_name")
            ).alias("customer_name"),
            
            # Standardize segment from variants: segment, customer_segment
            coalesce(
                col("segment"),
                col("customer_segment")
            ).alias("segment"),
            
            # These columns are already standardized
            col("country"),
            col("city"),
            col("state"),
            col("postal_code"),
            col("region"),
            col("source_region"),
            col("_rescued_data"), 
            col("_source_file_path"), 
            col("_source_file_name"), 
            col("_source_modified_time"), 
            col("_ingested_at")
        ))
    df = df.withColumn(
      "region",
      when(col("region").isin("N", "North"), "North")
      .when(col("region").isin("S", "South"), "South")
      .when(col("region").isin("E", "East"), "East")
      .when(col("region").isin("W", "West"), "West")
      .when(col("region").isin("C", "Central"), "Central")
    )
    df = df.withColumn(
      "segment",
      when(col("segment").isin("CONS", "Consumer", "Cosumer"), "Consumer")
      .when(col("segment").isin("CORP", "Corporate"), "Corporate")
      .when(col("segment").isin("HO", "Home Office"), "Home Office"))
    return df

In [0]:
@dp.temporary_view(
    name="orders_bronze_view",
    comment="Standardized view of bronze customers with unified column names"
)
def orders_bronze_view():
    """
    Temporary view that standardizes customer data from bronze layer.
    Handles multiple column name variants using COALESCE to merge into standard schema.
    """
    df = spark.readStream.table("orders")
    df = df.withColumn("order_approved_at", to_date(col("order_approved_at")))
    df = df.withColumn("order_delivered_carrier_date", to_date(col("order_delivered_carrier_date")))
    df = df.withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date")))
    df = df.withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))
    df = df.withColumn(
      "order_status",
      when(col("order_status").isin("canceled", "Cancelled"), "Cancelled")
      .when(col("order_status").isin("created", "Created"), "Created")
      .when(col("order_status").isin("delivered", "Delivered"), "Delivered")
      .when(col("order_status").isin("invoiced", "Invoiced"), "Invoiced")
      .when(col("order_status").isin("processing", "Processing"), "Processing")
      .when(col("order_status").isin("shipped", "Shipped"), "Shipped")
      .when(col("order_status").isin("unavailable", "Unavailable"), "Unavailable")
    )
    df = df.withColumn(
      "ship_mode",
      when(col("ship_mode") == "1st Class", "First Class")
      .when(col("ship_mode") == "2nd Class", "Second Class")
      .when(col("ship_mode") == "Std Class", "Standard Class")
      .otherwise(initcap(trim(col("ship_mode"))))
    )
    return df



In [0]:
@dp.temporary_view(
    name="returns_bronze_view",
    comment="Standardized view of bronze returns with unified column names"
)
def returns_bronze_view():
    """
    Temporary view that standardizes returns data from bronze layer.
    Handles multiple column name variants using COALESCE to merge into standard schema.
    """
    df =  (
        spark.readStream.table("returns")
        .select(
            # Standardize order_id from variants: order_id, OrderId
            coalesce(
                col("order_id"),
                col("OrderId")
            ).alias("order_id"),
            
            # Standardize return_reason from variants: return_reason, reason
            coalesce(
                col("return_reason"),
                col("reason")
            ).alias("return_reason"),
            
            # Standardize return_date from variants: return_date, date_of_return
            coalesce(
                col("return_date"),
                col("date_of_return")
            ).alias("return_date"),
            
            # Standardize refund_amount from variants: refund_amount, amount
            coalesce(
                col("refund_amount"),
                col("amount")
            ).alias("refund_amount"),
            
            # Standardize return_status from variants: return_status, status
            coalesce(
                col("return_status"),
                col("status")
            ).alias("return_status"),
            col("_rescued_data"), 
            col("_source_file_path"), 
            col("_source_file_name"), 
            col("_source_modified_time"), 
            col("_ingested_at")
        )
    )
    df = df.withColumn("return_date", to_date(col("return_date")))
    df = df.withColumn(
      "return_reason",
      when(col("return_reason") == "NULL", None)
      .otherwise(trim(col("return_reason")))
    )
    df = df.withColumn(
      "return_reason",
      when(col("return_reason") == "?", None)
      .otherwise(col("return_reason"))
    )
    df =df.withColumn(
  "refund_amount",
  regexp_replace(col("refund_amount"), "[$,]", "").cast("double")
    )
    return df


In [0]:
@dp.temporary_view(
    name="transactions_bronze_view",
    comment="Standardized view of bronze transactions with retrieved column names from _rescued_data column"
)
def transactions_bronze_view():
    df = spark.readStream.table("transactions")
    df = df.withColumn("rescued_map", from_json(col("_rescued_data"), MapType(StringType(), StringType())))
    df = df.withColumns({
                            "order_id": when(
                                col("_rescued_data").isNotNull(),
                                coalesce(
                                    col("order_id"),
                                    col("rescued_map")["Order_ID"]
                                )
                            ).otherwise(col("order_id")),

                            "product_id": when(
                                col("_rescued_data").isNotNull(),
                                coalesce(
                                    col("product_id"),
                                    col("rescued_map")["Product_ID"]
                                )
                            ).otherwise(col("product_id"))
                            })

    df = df.drop("rescued_map")

    return df